# Experiment no 2

## 1. Aim
To apply Principal Component Analysis (PCA) and Linear Discriminant Analysis (LDA) on a suitable open-access high-dimensional dataset and observe the influence of dimensionality reduction on classification performance and computational efficiency.

## 2. Objectives
1. Understand dimensionality reduction and its importance in machine learning.
2. Load and explore an open-access high-dimensional dataset.
3. Apply PCA, an unsupervised dimensionality reduction technique.
4. Apply LDA, a supervised dimensionality reduction technique.
5. Train and evaluate a classifier on original and reduced feature sets.
6. Compare model performance, dimensionality, and computational time.
7. Visualize PCA and LDA projections and interpret the results.

## 3. Software and Hardware Requirements
- **Language:** Python 3.x
- **Platform:** Google Colab / Jupyter Notebook
- **Libraries:** Pandas, NumPy, Matplotlib, Scikit-learn
- **Hardware:** Computer with at least 4 GB RAM

## 4. Dataset
This experiment uses the **Breast Cancer Wisconsin (Diagnostic)** dataset available through Scikit-learn.
- Samples: 569
- Input features: 30 numerical features
- Target classes: 2 (malignant, benign)
- Loaded directly via `sklearn.datasets.load_breast_cancer`.

## 5. Theory
### 5.1 Dimensionality Reduction
Dimensionality reduction reduces the number of input features while attempting to retain useful information. It reduces computational cost, simplifies models, reduces noise, and supports visualization.

### 5.2 Principal Component Analysis (PCA)
PCA is an **unsupervised** linear dimensionality reduction technique. It transforms correlated input features into a smaller number of principal components ordered by the amount of variance they explain. PCA does not use class labels.

### 5.3 Linear Discriminant Analysis (LDA)
LDA is a **supervised** dimensionality reduction technique that uses class labels to find directions that maximize class separability. For $C$ classes, LDA can produce at most $C - 1$ discriminant components (1 component for binary classification).

### 5.4 Standardization
PCA and LDA are sensitive to feature scales. Numerical features must be standardized using `StandardScaler` fitted on training data only to prevent data leakage.

## 6. Experimental Design
The same train/test split and Logistic Regression classifier are used to compare:
1. Original 30 standardized features.
2. PCA-reduced features (retaining 95% of training-set variance).
3. PCA with 2 components (for visualization and aggressive reduction).
4. LDA-reduced features (1 component for two-class dataset).


### Step 1: Import Libraries


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


### Step 2: Load and Explore the Dataset


In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()

print("Dataset shape:", X.shape)
print("Number of samples:", X.shape[0])
print("Number of input features:", X.shape[1])
print("Target classes:", dict(enumerate(dataset.target_names)))

print("\nFirst five records:")
display(X.head())

print("\nClass distribution:")
display(y.map({i: name for i, name in enumerate(dataset.target_names)})
        .value_counts().rename_axis("Class").to_frame("Count"))

print("\nMissing values:", int(X.isnull().sum().sum()))


### Step 3: Split Data and Standardize Features


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Original feature count:", X_train.shape[1])
print("Training scaled matrix shape:", X_train_scaled.shape)
print("Testing scaled matrix shape:", X_test_scaled.shape)


### Step 4: Define Evaluation Function and Baseline Model


In [ ]:
results = []

def evaluate_model(model_name, Xtr, Xte, ytr, yte, method_name):
    model = LogisticRegression(max_iter=5000, random_state=42)
    
    start_fit = time.perf_counter()
    model.fit(Xtr, ytr)
    fit_seconds = time.perf_counter() - start_fit
    
    start_pred = time.perf_counter()
    y_pred = model.predict(Xte)
    predict_seconds = time.perf_counter() - start_pred
    
    row = {
        "Method": method_name,
        "Features": Xtr.shape[1],
        "Accuracy": accuracy_score(yte, y_pred),
        "Precision (macro)": precision_score(yte, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(yte, y_pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(yte, y_pred, average="macro", zero_division=0),
        "Fit time (s)": fit_seconds,
        "Prediction time (s)": predict_seconds
    }
    results.append(row)
    return model, y_pred

baseline_model, baseline_pred = evaluate_model(
    "Logistic Regression", X_train_scaled, X_test_scaled,
    y_train, y_test, "Original features (30)"
)

print("Baseline model trained on all 30 standardized features.")
print("\nClassification report:")
print(classification_report(
    y_test, baseline_pred,
    target_names=dataset.target_names,
    zero_division=0
))


### Step 5: Apply PCA — Retain 95% of Variance


In [ ]:
pca_95 = PCA(n_components=0.95, svd_solver="full")

start_fit = time.perf_counter()
X_train_pca95 = pca_95.fit_transform(X_train_scaled)
pca_fit_seconds = time.perf_counter() - start_fit
X_test_pca95 = pca_95.transform(X_test_scaled)

print("Original feature count:", X_train_scaled.shape[1])
print("PCA component count:", pca_95.n_components_)
print("Training reduced shape:", X_train_pca95.shape)
print("Testing reduced shape:", X_test_pca95.shape)
print("Total explained variance:", round(pca_95.explained_variance_ratio_.sum(), 4))
print("PCA fit time (s):", round(pca_fit_seconds, 6))

pca95_model, pca95_pred = evaluate_model(
    "Logistic Regression", X_train_pca95, X_test_pca95,
    y_train, y_test, f"PCA (95% variance, {pca_95.n_components_} components)"
)

print("\nClassification report after PCA (95% variance):")
print(classification_report(
    y_test, pca95_pred,
    target_names=dataset.target_names,
    zero_division=0
))


### Step 6: Apply PCA with 2 Components


In [ ]:
pca_2 = PCA(n_components=2)

X_train_pca2 = pca_2.fit_transform(X_train_scaled)
X_test_pca2 = pca_2.transform(X_test_scaled)

print("PCA components:", pca_2.n_components_)
print("Explained variance ratio:", np.round(pca_2.explained_variance_ratio_, 4))
print("Total variance retained:", round(pca_2.explained_variance_ratio_.sum(), 4))

pca2_model, pca2_pred = evaluate_model(
    "Logistic Regression", X_train_pca2, X_test_pca2,
    y_train, y_test, "PCA (2 components)"
)

print("\nClassification report after PCA (2 components):")
print(classification_report(
    y_test, pca2_pred,
    target_names=dataset.target_names,
    zero_division=0
))


### Step 7: Apply LDA


In [ ]:
# There are two classes, so LDA can produce at most 2 - 1 = 1 component.
lda = LinearDiscriminantAnalysis(n_components=1)

start_fit = time.perf_counter()
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
lda_fit_seconds = time.perf_counter() - start_fit
X_test_lda = lda.transform(X_test_scaled)

print("Original feature count:", X_train_scaled.shape[1])
print("LDA component count:", X_train_lda.shape[1])
print("Training reduced shape:", X_train_lda.shape)
print("Testing reduced shape:", X_test_lda.shape)
print("LDA fit time (s):", round(lda_fit_seconds, 6))

lda_model, lda_pred = evaluate_model(
    "Logistic Regression", X_train_lda, X_test_lda,
    y_train, y_test, "LDA (1 component)"
)

print("\nClassification report after LDA:")
print(classification_report(
    y_test, lda_pred,
    target_names=dataset.target_names,
    zero_division=0
))


### Step 8: Visualize PCA and LDA Projections


In [ ]:
# PCA 2D scatter plot (training set)
plt.figure(figsize=(8, 6))
for class_value, class_name in enumerate(dataset.target_names):
    mask = (y_train.to_numpy() == class_value)
    plt.scatter(
        X_train_pca2[mask, 0],
        X_train_pca2[mask, 1],
        label=class_name,
        alpha=0.7
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Projection of Training Data")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# LDA one-dimensional projection
plt.figure(figsize=(8, 4))
for class_value, class_name in enumerate(dataset.target_names):
    mask = (y_train.to_numpy() == class_value)
    plt.scatter(
        X_train_lda[mask, 0],
        np.zeros(mask.sum()) + class_value * 0.04,
        label=class_name,
        alpha=0.7
    )

plt.xlabel("Linear Discriminant 1")
plt.yticks([])
plt.title("LDA Projection of Training Data")
plt.legend()
plt.grid(True, axis="x", alpha=0.25)
plt.show()


### Step 9: Compare Model Performance and Computational Efficiency


In [ ]:
results_df = pd.DataFrame(results)

original_features = X_train_scaled.shape[1]
results_df["Feature reduction (%)"] = (
    (original_features - results_df["Features"]) / original_features * 100
).round(2)

# Round displayed metrics for readability
display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision (macro)": "{:.4f}",
        "Recall (macro)": "{:.4f}",
        "F1-score (macro)": "{:.4f}",
        "Fit time (s)": "{:.6f}",
        "Prediction time (s)": "{:.6f}",
        "Feature reduction (%)": "{:.2f}%"
    })
)

print("\nNote: Timing measurements depend on hardware, software, and system load.")


#### Plot Accuracy and Feature Count


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(results_df["Method"], results_df["Accuracy"])
axes[0].set_title("Test Accuracy by Method")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(results_df["Method"], results_df["Features"])
axes[1].set_title("Number of Input Features")
axes[1].set_ylabel("Feature count")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


#### Plot Fit and Prediction Times


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(results_df["Method"], results_df["Fit time (s)"])
axes[0].set_title("Classifier Fit Time")
axes[0].set_ylabel("Seconds")
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(results_df["Method"], results_df["Prediction time (s)"])
axes[1].set_title("Classifier Prediction Time")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


### Step 10: Confusion Matrix for Each Method


In [ ]:
predictions = [
    ("Original features (30)", baseline_pred),
    (f"PCA (95% variance, {pca_95.n_components_} components)", pca95_pred),
    ("PCA (2 components)", pca2_pred),
    ("LDA (1 component)", lda_pred)
]

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, y_pred) in zip(axes.ravel(), predictions):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=dataset.target_names
    )
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()


## 11. Observations and Discussion
1. **Feature reduction:** PCA with 95% variance generally reduces features from 30 down to ~10. PCA with 2 components and LDA with 1 component reduce dimensionality even more aggressively (up to 96% reduction).
2. **Classification performance:** Retained features preserve most predictive information; accuracy remains high (~97-98%).
3. **Computational efficiency:** Lower feature counts reduce downstream model fitting and prediction costs.
4. **PCA versus LDA:** PCA preserves directions of highest variance without labels; LDA uses labels to seek class-separating directions.
5. **Fair comparison:** The same train/test split and classifier are used across experiments.

## 12. Result
PCA and LDA were applied to the 30-feature Breast Cancer Wisconsin dataset. Logistic Regression models were trained on the original and reduced feature representations. Classification metrics, feature counts, and timing measurements were collected to observe the effect of dimensionality reduction on predictive performance and computational efficiency.

## 13. Conclusion
This experiment demonstrates two dimensionality reduction techniques. PCA compresses the feature space by preserving directions with high variance, while LDA uses class labels to find a discriminative projection. Comparing model performance and runtime shows the trade-off between reducing input dimensions and retaining useful predictive information.

## 14. Viva Voce Questions and Answers

**Q1. What is dimensionality reduction?**  
*Answer:* It is the process of reducing the number of features while attempting to preserve useful information.

**Q2. What is PCA?**  
*Answer:* PCA is an unsupervised technique that creates orthogonal components ordered by the amount of variance they explain.

**Q3. What is LDA?**  
*Answer:* LDA is a supervised technique that uses class labels to find directions that maximize between-class variance relative to within-class variance.

**Q4. What is the main difference between PCA and LDA?**  
*Answer:* PCA does not use class labels (unsupervised); LDA uses class labels (supervised).

**Q5. Why is scaling important before PCA?**  
*Answer:* Scaling prevents features with large numerical ranges from dominating the variance calculation simply because of their units.

**Q6. How many LDA components can be obtained for a two-class problem?**  
*Answer:* At most one component, because the maximum number of components is $C - 1$ where $C$ is the number of classes ($2 - 1 = 1$).

**Q7. Does dimensionality reduction always improve accuracy?**  
*Answer:* No. It may remove noise and improve efficiency, but it can also discard useful predictive information if too few components are kept.

**Q8. How can computational efficiency be evaluated?**  
*Answer:* By comparing model fit time, prediction time, feature count, and memory usage across different feature representations.
